# MVP - Data Engineering

In [0]:
from pyspark.sql.functions import col, count, when, isnull, sum as spark_sum, avg as spark_avg, row_number, count as count_all, expr, min as spark_min, max as spark_max, percentile_approx, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType, DateType, BooleanType
from pyspark.sql.window import Window

from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

## Coleta

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp_pucrio.bronze_ecommerce;

In [0]:
schemas = {
    "customers": StructType([
        StructField("customer_id", StringType(), False),
        StructField("customer_unique_id", StringType(), True),
        StructField("customer_zip_code_prefix", IntegerType(), True),
        StructField("customer_city", StringType(), True),
        StructField("customer_state", StringType(), True)
    ]),
    
    "orders": StructType([
        StructField("order_id", StringType(), False),
        StructField("customer_id", StringType(), False),
        StructField("order_status", StringType(), True),
        StructField("order_purchase_timestamp", TimestampType(), True),
        StructField("order_approved_at", TimestampType(), True),
        StructField("order_delivered_carrier_date", TimestampType(), True),
        StructField("order_delivered_customer_date", TimestampType(), True),
        StructField("order_estimated_delivery_date", TimestampType(), True)
    ]),
    
    "order_items": StructType([
        StructField("order_id", StringType(), False),
        StructField("order_item_id", IntegerType(), True),
        StructField("product_id", StringType(), False),
        StructField("seller_id", StringType(), False),
        StructField("shipping_limit_date", TimestampType(), True),
        StructField("price", DoubleType(), True),
        StructField("freight_value", DoubleType(), True)
    ]),
    
    "products": StructType([
        StructField("product_id", StringType(), False),
        StructField("product_category_name", StringType(), True),
        StructField("product_name_lenght", IntegerType(), True),
        StructField("product_description_lenght", IntegerType(), True),
        StructField("product_photos_qty", IntegerType(), True),
        StructField("product_weight_g", IntegerType(), True),
        StructField("product_length_cm", IntegerType(), True),
        StructField("product_height_cm", IntegerType(), True),
        StructField("product_width_cm", IntegerType(), True)
    ]),
    
    "sellers": StructType([
        StructField("seller_id", StringType(), False),
        StructField("seller_zip_code_prefix", IntegerType(), True),
        StructField("seller_city", StringType(), True),
        StructField("seller_state", StringType(), True)
    ]),
    
    "marketing_qualified_leads": StructType([
        StructField("mql_id", StringType(), False),
        StructField("first_contact_date", DateType(), True),
        StructField("landing_page_id", StringType(), True),
        StructField("origin", StringType(), True)
    ]),
    
    "closed_deals": StructType([
        StructField("mql_id", StringType(), False),
        StructField("seller_id", StringType(), True),
        StructField("sdr_id", StringType(), True),
        StructField("sr_id", StringType(), True),
        StructField("won_date", TimestampType(), True),
        StructField("business_segment", StringType(), True),
        StructField("lead_type", StringType(), True),
        StructField("lead_behaviour_profile", StringType(), True),
        StructField("has_company", BooleanType(), True),
        StructField("has_gtin", BooleanType(), True),
        StructField("average_stock", StringType(), True),
        StructField("business_type", StringType(), True),
        StructField("declared_product_catalog_size", DoubleType(), True),
        StructField("declared_monthly_revenue", DoubleType(), True)
    ])
}

In [0]:
volume_path = "/Volumes/mvp_pucrio/raw_files/csv_files"
catalog = "mvp_pucrio.bronze_ecommerce."

files = dbutils.fs.ls(volume_path)

for fl in files:
    fl = fl.name
    if fl.endswith('.csv'):
        fl_name = fl.split('.')[0]
        fl_name = fl_name.replace('olist_', '').replace('_dataset', '')
        
        # Usar schema explícito ao invés de inferSchema
        schema = schemas.get(fl_name)
        if schema is None:
            print(f"- Schema não encontrado para {fl_name}, pulando arquivo...")
            continue
        
        df_orders = spark.read.csv(
            f"{volume_path}/{fl}",
            header=True,
            schema=schema,
            multiLine=True
        )
        
        df_orders = (df_orders
            .withColumn("_source_file", lit(fl))
            .withColumn("_ingested_at", current_timestamp())
        )

        df_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
            catalog + fl_name
        )
        
        print(f"- {fl_name} carregado com schema explícito")

## Diagnóstico de Dados

In [0]:
# Primary keys do modelo (Bronze, Silver E Gold)
pk_dict = {
    # Bronze/Silver
    "customers": "customer_id",
    "sellers": "seller_id",
    "marketing_qualified_leads": "mql_id",
    "closed_deals": "mql_id",
    "products": "product_id",
    "orders": "order_id",
    "order_items": ["order_id", "product_id", "seller_id"],  # PK composta
    # Gold
    "dim_leads": "mql_id",
    "dim_customers": "customer_id",
    "dim_sellers": "seller_id",
    "dim_produtos": "product_id",
    "dim_dates": "order_date",
    "fato_vendas": ["order_id", "product_id", "seller_id"],  # PK composta
}

# Todas as colunas do modelo (Bronze, Silver E Gold)
columns_to_model = {
    "mql_id", "customer_id", "seller_id", "product_id", "order_id",
    "first_contact_date", "origin",
    "won_date", "sales_cycle",
    "business_segment", "lead_type",
    "customer_city", "customer_state",
    "seller_city", "seller_state",
    "price", "sales_value", "commission", "quantity",
    "product_category_name", "category",
    "order_purchase_timestamp", "order_date", "time_category",
    "year", "month", "quarter", "day_of_week",
    "order_status"
}

In [0]:
def find_duplicates(df, pk_dict, columns_to_model, table_name, num_nulls, problems_found, duplicates_set):
    """Acha linhas duplicadas e colunas PK duplicadas no dataframe."""
    
    df_cols = [c for c in df.columns if c in columns_to_model]
    total_rows = df.count()
    distinct_rows = df.select(df_cols).distinct().count()
    duplicate_count = total_rows - distinct_rows
    
    # Duplicated rows table
    w = Window.partitionBy([col(c) for c in df_cols])
    duplicated_rows = df.withColumn("_dup_count", count_all("*").over(w)) \
        .filter(col("_dup_count") > 1) \
        .drop("_dup_count")
    
    # PK column duplicates
    pk_col = pk_dict.get(table_name)
    pk_dup_count = 0
    if pk_col:
        pk_cols_list = pk_col if isinstance(pk_col, list) else [pk_col]
        if all(c in df_cols for c in pk_cols_list):
            pk_dup_count = total_rows - df.select(pk_cols_list).distinct().count()

    if duplicate_count > 0 or pk_dup_count > 0:
        if num_nulls == 0:
            print(f"\n{'='*60}")
            print(f"Table: {table_name}")
        
        duplicates_set.add(table_name)
        print(f"\nDuplicados:")
        print(f"Linhas: {total_rows} | Duplicados: {duplicate_count} | PK duplicados: {pk_dup_count}")

        print(f"\nLinhas duplicadas em {table_name}:")
        duplicated_rows.orderBy("order_id").limit(20).show(truncate=False)

        problems_found += 1
    
    return problems_found, duplicates_set

In [0]:
def find_nulls(df, pk_list, columns_to_model, problems_found, nulls_dict):
    """Count nulls per column and return rows containing nulls."""
    
    columns = [c for c in df.columns if c in columns_to_model]
    
    nulls_count = df.select([
        spark_sum(when(isnull(c), 1).otherwise(0)).alias(c) for c in columns
    ]).collect()[0]

    num_nulls = 0
    for col_name, null_count in nulls_count.asDict().items():
        if null_count > 0:
            if name not in nulls_dict.keys():
                nulls_dict[name] = [col_name]
            else:
                nulls_dict[name].append(col_name)

            if num_nulls == 0:
                print(f"\n{'='*60}")
                print(f"Table: {name}")
                print(f"\nNulls por coluna:")
            num_nulls += null_count
            print(f"  {col_name}: {null_count}")

            problems_found += 1
    
    return num_nulls, problems_found, nulls_dict

In [0]:
def schema_tables(schema):
    tables = spark.sql(f"SHOW TABLES IN {schema}").collect()
    return [row.tableName for row in tables]

In [0]:
nulls_dict = dict()
duplicates_set = set()

print("Diagnóstico de Dados - Tabelas que requerem limpeza (bronze):")
problems = 0

schema = "mvp_pucrio.bronze_ecommerce"
table_names = schema_tables(schema)

for name in table_names:
    full_name = f"{schema}.{name}"
    df = spark.table(full_name)

    num_nulls, problems, nulls_dict = find_nulls(df, pk_dict, columns_to_model, problems, nulls_dict)
    
    # Duplicates
    problems, duplicates_set = find_duplicates(df, pk_dict, columns_to_model, name, num_nulls, problems, duplicates_set)

if problems == 0:
    print("Nenhuma tabela apresenta valores nulos ou duplicados.")

## Carga dos dados (ETL)

### Bronze → Silver

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp_pucrio.silver_ecommerce;

In [0]:
def removing_cols(df):
    """Remove colunas que não fazem parte do modelo."""
    cols_to_remove = set(df.columns) - columns_to_model

    return df.drop(*cols_to_remove)

In [0]:
for table in table_names:
    df = spark.table(f"mvp_pucrio.bronze_ecommerce.{table}")
    df = removing_cols(df)
    
    if table == "order_items":
        df = df.groupBy(["order_id", "product_id", "seller_id"]).agg(
            count_all("*").alias("quantity"),
            spark_sum("price").alias("sales_value")
        )

    if table in nulls_dict.keys():
        df = df.fillna("unknown", subset=nulls_dict[table])
    
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"mvp_pucrio.silver_ecommerce.{table}")

### Verificando limpeza das tabelas

In [0]:
nulls_dict = dict()
duplicates_set = set()

print("Diagnóstico de Dados - Tabelas que requerem limpeza (silver):")
problems = 0

schema = "mvp_pucrio.silver_ecommerce"
table_names = schema_tables(schema)

for name in table_names:
    full_name = f"{schema}.{name}"
    df = spark.table(full_name)

    num_nulls, problems, nulls_dict = find_nulls(df, pk_dict, columns_to_model, problems, nulls_dict)
    
    # Duplicates
    problems, duplicates_set = find_duplicates(df, pk_dict, columns_to_model, name, num_nulls, problems, duplicates_set)

if problems == 0:
    print("Nenhuma tabela apresenta valores nulos ou duplicados.")

### Silver → Gold

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS mvp_pucrio.gold_ecommerce;

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_pucrio.gold_ecommerce.dim_leads AS
SELECT 
    mql.mql_id,
    cd.seller_id,
    mql.origin,
    mql.first_contact_date,
    CAST(cd.won_date AS DATE) AS won_date,
    DATEDIFF(CAST(cd.won_date AS DATE), mql.first_contact_date) AS sales_cycle,
    COALESCE(cd.business_segment, 'unknown') AS business_segment,
    COALESCE(cd.lead_type, 'unknown') AS lead_type
FROM mvp_pucrio.silver_ecommerce.marketing_qualified_leads mql
LEFT JOIN mvp_pucrio.silver_ecommerce.closed_deals cd
    ON mql.mql_id = cd.mql_id;

In [0]:
%sql
COMMENT ON TABLE mvp_pucrio.gold_ecommerce.dim_leads IS 
'Dimensão consolidada de TODOS os leads qualificados de marketing (MQLs) - tanto convertidos quanto não convertidos. Combina dados de marketing_qualified_leads com closed_deals via LEFT JOIN para facilitar análises de funil completo. Leads não convertidos terão seller_id, won_date, sales_cycle, business_segment e lead_type como NULL. Uso: análises de funil de conversão, taxa de conversão por canal, ciclo de vendas, performance de segmentos.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.mql_id IS 
'Primary Key - ID único do lead qualificado de marketing. Tipo: STRING. Justificativa: mql_id é sempre único (cada lead é um ponto de entrada distinto), enquanto seller_id pode repetir se um seller veio de múltiplos leads.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.seller_id IS 
'ID do seller resultante da conversão. Tipo: STRING. NULL para leads não convertidos. Corresponde a dim_sellers.seller_id quando não-nulo.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.origin IS 
'Canal de origem do lead (organic_search, paid_search, social, email, direct ou "unknown"). Tipo: STRING. Usado para análise de ROI por canal e taxa de conversão por origem.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.first_contact_date IS 
'Data do primeiro contato com o lead. Tipo: DATE. Usado para calcular sales_cycle e análises de coorte temporal.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.won_date IS 
'Data de conversão (fechamento do negócio). Tipo: DATE. NULL para leads não convertidos. Usado para calcular sales_cycle e análises de conversão ao longo do tempo.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.sales_cycle IS 
'Dias entre primeiro contato e conversão. Tipo: INT. Fórmula: DATEDIFF(won_date, first_contact_date). NULL para leads não convertidos. Métrica chave para eficiência do funil de vendas.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.business_segment IS 
'Segmento de negócio do seller (pet, health_beauty, electronics, etc). Tipo: STRING. "unknown" para leads não convertidos. Usado para segmentação de análises por setor.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_leads.lead_type IS 
'Tipo/tamanho do lead (online_small, online_medium, online_big, etc). Tipo: STRING. "unknown" para leads não convertidos. Usado para análise de ticket médio e LTV por tipo de lead.';

In [0]:
%sql
ALTER TABLE mvp_pucrio.gold_ecommerce.dim_leads 
ALTER COLUMN mql_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.dim_leads 
ADD CONSTRAINT pk_dim_leads PRIMARY KEY (mql_id);

In [0]:
%sql
SELECT * FROM mvp_pucrio.gold_ecommerce.dim_leads LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_pucrio.gold_ecommerce.dim_customers AS
SELECT 
    customer_id,
    customer_city,
    customer_state
FROM mvp_pucrio.silver_ecommerce.customers;

In [0]:
%sql
COMMENT ON TABLE mvp_pucrio.gold_ecommerce.dim_customers IS 
'Dimensão de clientes que realizaram compras. Origem: silver.customers. Uso: análise geográfica de clientes, segmentação regional, concentração de receita por região.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_customers.customer_id IS 
'Primary Key - ID único do cliente. Tipo: STRING.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_customers.customer_city IS 
'Cidade de residência do cliente. Tipo: STRING. Valores: Cidades brasileiras ou "unknown".';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_customers.customer_state IS 
'Estado (UF) de residência do cliente. Tipo: STRING. Valores: SP, RJ, MG, etc ou "unknown". Usado para análise de receita por região.';

In [0]:
%sql
ALTER TABLE mvp_pucrio.gold_ecommerce.dim_customers 
ALTER COLUMN customer_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.dim_customers 
ADD CONSTRAINT pk_dim_customers PRIMARY KEY (customer_id);

In [0]:
%sql
SELECT * FROM mvp_pucrio.gold_ecommerce.dim_customers LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_pucrio.gold_ecommerce.dim_sellers AS
SELECT 
    seller_id,
    seller_city,
    seller_state
FROM mvp_pucrio.silver_ecommerce.sellers;

In [0]:
%sql
COMMENT ON TABLE mvp_pucrio.gold_ecommerce.dim_sellers IS 
'Dimensão de sellers (vendedores) ativos na plataforma. Origem: silver.sellers. Uso: análise geográfica de sellers, performance por localização, concentração de vendedores.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_sellers.seller_id IS 
'Primary Key - ID único do seller. Tipo: STRING.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_sellers.seller_city IS 
'Cidade onde o seller opera. Tipo: STRING. Valores: Cidades brasileiras ou "unknown".';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_sellers.seller_state IS 
'Estado (UF) onde o seller opera. Tipo: STRING. Valores: SP, RJ, MG, etc ou "unknown". Usado para análise de concentração de sellers.';

In [0]:
%sql
ALTER TABLE mvp_pucrio.gold_ecommerce.dim_sellers 
ALTER COLUMN seller_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.dim_sellers 
ADD CONSTRAINT pk_dim_sellers PRIMARY KEY (seller_id);

In [0]:
%sql
SELECT * FROM mvp_pucrio.gold_ecommerce.dim_sellers LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_pucrio.gold_ecommerce.dim_produtos AS
SELECT DISTINCT
    product_id,
    product_category_name AS category
FROM mvp_pucrio.silver_ecommerce.products;

In [0]:
%sql
COMMENT ON TABLE mvp_pucrio.gold_ecommerce.dim_produtos IS 
'Dimensão de produtos disponíveis na plataforma. Origem: silver.products. Uso: análise de receita por categoria, mix de produtos. IMPORTANTE: Esta dimensão contém apenas atributos descritivos (category). Métricas como sales_value estão na fato_vendas.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_produtos.product_id IS 
'Primary Key - ID único do produto. Tipo: STRING.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_produtos.category IS 
'Categoria do produto (eletrônicos, cama_mesa_banho, beleza_saúde, brinquedos, automotivo, etc ou "unknown"). Tipo: STRING. Origem: silver.products.product_category_name.';

In [0]:
%sql
ALTER TABLE mvp_pucrio.gold_ecommerce.dim_produtos 
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.dim_produtos 
ADD CONSTRAINT pk_dim_produtos PRIMARY KEY (product_id);

In [0]:
%sql
SELECT * FROM mvp_pucrio.gold_ecommerce.dim_produtos LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_pucrio.gold_ecommerce.dim_dates AS
SELECT DISTINCT
    CAST(order_purchase_timestamp AS DATE) AS order_date,
    YEAR(order_purchase_timestamp) AS year,
    MONTH(order_purchase_timestamp) AS month,
    QUARTER(order_purchase_timestamp) AS quarter,
    DAYOFWEEK(order_purchase_timestamp) AS day_of_week
FROM mvp_pucrio.silver_ecommerce.orders
WHERE order_purchase_timestamp IS NOT NULL;

In [0]:
%sql
COMMENT ON TABLE mvp_pucrio.gold_ecommerce.dim_dates IS 
'Dimensão temporal para análises por período. Origem: DISTINCT order_purchase_timestamp de silver.orders. Uso: análises de tendência, sazonalidade, performance por trimestre/mês/dia da semana.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_dates.order_date IS 
'Primary Key - Data da transação. Tipo: DATE. Formato: YYYY-MM-DD. Valores: todas as datas únicas de compras.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_dates.year IS 
'Ano extraído da order_date. Tipo: INT. Valores: 2016-2018. Fórmula: YEAR(order_purchase_timestamp).';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_dates.month IS 
'Mês extraído da order_date. Tipo: INT. Valores: 1-12. Fórmula: MONTH(order_purchase_timestamp). Usado para análise de sazonalidade.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_dates.quarter IS 
'Trimestre do ano. Tipo: INT. Valores: 1-4. Fórmula: QUARTER(order_purchase_timestamp). Usado para relatórios trimestrais.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.dim_dates.day_of_week IS 
'Dia da semana. Tipo: INT. Valores: 1-7 (1=Domingo, 7=Sábado). Fórmula: DAYOFWEEK(order_purchase_timestamp). Usado para análise de padrões semanais.';

In [0]:
%sql
ALTER TABLE mvp_pucrio.gold_ecommerce.dim_dates 
ALTER COLUMN order_date SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.dim_dates 
ADD CONSTRAINT pk_dim_dates PRIMARY KEY (order_date);

In [0]:
%sql
SELECT * FROM mvp_pucrio.gold_ecommerce.dim_dates LIMIT 10;

In [0]:
%sql
CREATE OR REPLACE TABLE mvp_pucrio.gold_ecommerce.fato_vendas AS
SELECT 
    o.order_id,
    COALESCE(c.mql_id, "unknown") AS mql_id,
    oi.seller_id,
    o.customer_id,
    oi.product_id,
    CAST(o.order_purchase_timestamp AS DATE) AS order_date,
    CASE
        WHEN HOUR(o.order_purchase_timestamp) < 2 THEN '00:00 - 02:00'
        WHEN HOUR(o.order_purchase_timestamp) < 4 THEN '02:00 - 04:00'
        WHEN HOUR(o.order_purchase_timestamp) < 6 THEN '04:00 - 06:00'
        WHEN HOUR(o.order_purchase_timestamp) < 8 THEN '06:00 - 08:00'
        WHEN HOUR(o.order_purchase_timestamp) < 10 THEN '08:00 - 10:00'
        WHEN HOUR(o.order_purchase_timestamp) < 12 THEN '10:00 - 12:00'
        WHEN HOUR(o.order_purchase_timestamp) < 14 THEN '12:00 - 14:00'
        WHEN HOUR(o.order_purchase_timestamp) < 16 THEN '14:00 - 16:00'
        WHEN HOUR(o.order_purchase_timestamp) < 18 THEN '16:00 - 18:00'
        WHEN HOUR(o.order_purchase_timestamp) < 20 THEN '18:00 - 20:00'
        WHEN HOUR(o.order_purchase_timestamp) < 22 THEN '20:00 - 22:00'
        ELSE '22:00 - 24:00'
    END AS time_category,
    o.order_status,
    oi.quantity,
    oi.sales_value,
    ROUND(oi.sales_value * 0.10, 2) AS commission
FROM mvp_pucrio.silver_ecommerce.orders o
INNER JOIN mvp_pucrio.silver_ecommerce.order_items oi 
    ON o.order_id = oi.order_id
LEFT JOIN mvp_pucrio.silver_ecommerce.closed_deals c 
    ON oi.seller_id = c.seller_id;

In [0]:
%sql
COMMENT ON TABLE mvp_pucrio.gold_ecommerce.fato_vendas IS 
'Tabela FATO do modelo star schema. Cada linha representa uma transação de venda (item vendido). Origem: JOIN entre silver.orders e silver.order_items, com LEFT JOIN para silver.closed_deals. Contém métricas agregáveis (quantity, sales_value, comission) e chaves estrangeiras para todas as dimensões.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.order_id IS 
'Primary Key - ID único da ordem de venda. Tipo: STRING. Origem: silver.orders.order_id.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.mql_id IS 
'Foreign Key → dim_leads.mql_id. Identificador do lead qualificado de marketing associado ao seller desta venda. Tipo: STRING. Origem: silver.closed_deals.mql_id via LEFT JOIN por seller_id ("unknown" se seller não veio de lead rastreado).
ATENÇÃO: atribuição feita no nível do seller (via closed_deals.seller_id), não da venda individual — toda venda de um seller herda o canal de aquisição daquele seller.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.seller_id IS 
'Foreign Key → dim_sellers.seller_id. Identificador do vendedor que realizou a venda. Tipo: STRING. Origem: silver.order_items.seller_id. SEM NULOS.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.customer_id IS 
'Foreign Key → dim_customers.customer_id. Identificador do cliente que realizou a compra. Tipo: STRING. Origem: silver.orders.customer_id. SEM NULOS.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.product_id IS 
'Foreign Key → dim_produtos.product_id. Identificador do produto vendido. Tipo: STRING. Origem: silver.order_items.product_id. SEM NULOS.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.order_date IS 
'Foreign Key → dim_dates.order_date. Data da transação (sem hora) para análises temporais. Tipo: DATE. Formato: YYYY-MM-DD. Origem: CAST(silver.orders.order_purchase_timestamp AS DATE). SEM NULOS.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.time_category IS 
'Faixa horária da compra (categoria). Tipo: STRING. Valores: 00:00 - 02:00, 02:00 - 04:00, 04:00 - 06:00, 06:00 - 08:00, 08:00 - 10:00, 10:00 - 12:00, 12:00 - 14:00, 14:00 - 16:00, 16:00 - 18:00, 18:00 - 20:00, 20:00 - 22:00, 22:00 - 24:00. Origem: CASE sobre HOUR(order_purchase_timestamp). Usado para analise de padroes de compra por horario.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.order_status IS 
'Status do pedido (delivered, shipped, canceled, processing, etc). Tipo: STRING. Origem: silver.orders.order_status. SEM NULOS.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.quantity IS 
'MÉTRICA - Quantidade de itens do mesmo produto na ordem. Tipo: INT. Valores: >= 1. Origem: silver.order_items.quantity (COUNT de linhas agrupadas por order_id + product_id + seller_id). SEM NULOS. Agregável: SUM, AVG.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.sales_value IS 
'MÉTRICA - Valor total vendido nesta transação (em BRL). Tipo: DECIMAL. Valores: >= 0. Origem: silver.order_items.sales_value. SEM NULOS. Agregável: SUM, AVG.';

COMMENT ON COLUMN mvp_pucrio.gold_ecommerce.fato_vendas.commission IS 
'MÉTRICA - Comissão da plataforma (10% do sales_value). Tipo: DECIMAL. Valores: >= 0. Fórmula: ROUND(sales_value * 0.10, 2). SEM NULOS. Agregável: SUM, AVG.';

In [0]:
%sql
ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ALTER COLUMN order_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ALTER COLUMN product_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ALTER COLUMN seller_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ALTER COLUMN customer_id SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ALTER COLUMN order_date SET NOT NULL;

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ADD CONSTRAINT pk_fato_vendas PRIMARY KEY (order_id, product_id, seller_id);

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ADD CONSTRAINT fk_fato_leads FOREIGN KEY (mql_id) 
REFERENCES mvp_pucrio.gold_ecommerce.dim_leads(mql_id);

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ADD CONSTRAINT fk_fato_customers FOREIGN KEY (customer_id) 
REFERENCES mvp_pucrio.gold_ecommerce.dim_customers(customer_id);

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ADD CONSTRAINT fk_fato_sellers FOREIGN KEY (seller_id) 
REFERENCES mvp_pucrio.gold_ecommerce.dim_sellers(seller_id);

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ADD CONSTRAINT fk_fato_produtos FOREIGN KEY (product_id) 
REFERENCES mvp_pucrio.gold_ecommerce.dim_produtos(product_id);

ALTER TABLE mvp_pucrio.gold_ecommerce.fato_vendas 
ADD CONSTRAINT fk_fato_dates FOREIGN KEY (order_date) 
REFERENCES mvp_pucrio.gold_ecommerce.dim_dates(order_date);

In [0]:
%sql
SELECT * FROM mvp_pucrio.gold_ecommerce.fato_vendas LIMIT 10;

In [0]:
%sql
SELECT ROUND(SUM(f.sales_value) - SUM(oi.sales_value),2) AS checksum
FROM mvp_pucrio.silver_ecommerce.order_items oi 
JOIN mvp_pucrio.gold_ecommerce.fato_vendas f ON f.order_id = oi.order_id;

## Análise dos dados

### Qualidade dos dados

In [0]:
def check_state(df, table_name):
    """Verifica se os estados estão no conjunto de estados brasileiros, no formato especificado."""
    brazilian_states = ["AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO", "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI", "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO", "unknown"]

    if table_name == "dim_sellers":
        state = "seller_state"
    elif table_name == "dim_customers":
        state = "customer_state"
    else:
        state = "state"

    wrong_state = df.filter(~df[state].isin(brazilian_states))

    problems = 0
    if wrong_state.count() > 0:
        problems = 1
        print(f"\nQuantidade de linhas com estado inválido: {wrong_state.count()}")
        wrong_state.show(truncate=False)

    return problems

In [0]:
def check_shipment(df, table_name):
    """Verifica se status de entrega é válido."""

    valid_status = ["delivered", "shipped", "canceled", "processing", "unavailable", "invoiced", "created", "approved"]
    
    invalid_status = df.filter(~col("order_status").isin(valid_status))
    count = invalid_status.count()

    problems = 0
    if count > 0:
        problems = 1
        print(f"\nQuantidade de linhas com order_status desconhecido: {count}")
        invalid_status.select("order_status").distinct().show(truncate=False)
    
    return problems

In [0]:
def check_temporal_accuracy(df, table_name, last_date):
    """Verifica se datas fazem sentido no contexto do negócio."""
    problems = 0
    
    # 1. Datas no futuro (impossível)
    date_cols = ["order_date", "first_contact_date", "won_date"]
    for date_col in date_cols:
        if date_col in df.columns:
            future_dates = df.filter(col(date_col) > last_date)
            count = future_dates.count()
            if count > 0:
                problems = 1
                print(f"\nQuantidade de linhas com {date_col} no futuro: {count}")
                future_dates.limit(10).show(truncate=False)
    
    # 2. Won_date deve ser >= first_contact_date (para leads convertidos)
    if table_name == "dim_leads":
        # Filtrar apenas leads convertidos (com won_date)
        joined = df.filter(col("won_date").isNotNull())
        invalid_dates = joined.filter(
            (col("won_date").isNotNull()) & 
            (col("first_contact_date").isNotNull()) &
            (col("won_date") < col("first_contact_date"))
        )
        count = invalid_dates.count()
        if count > 0:
            problems += count
            print(f"\nQuantidade de linhas onde won_date < first_contact_date: {count}")
            invalid_dates.select("mql_id", "first_contact_date", "won_date").limit(10).show(truncate=False)
    
    return problems

In [0]:
def detect_outliers(df, table_name):
    """Detecta outliers de preço unitário (sales_value / quantity) por produto (IQR method).
    Compara preços unitários apenas dentro do mesmo product_id, evitando falsos outliers
    causados pela diferença natural de preço entre produtos distintos."""
    
    problems = 0
    
    if table_name == "fato_vendas":
        # Preço unitário = sales_value / quantity
        df = df.withColumn("unit_price", col("sales_value") / col("quantity"))
        
        # Estatísticas IQR por produto (baseado em unit_price)
        product_stats = df.groupBy("product_id").agg(
            percentile_approx("unit_price", 0.25).alias("q1"),
            percentile_approx("unit_price", 0.75).alias("q3"),
            percentile_approx("unit_price", 0.50).alias("median"),
            spark_avg("unit_price").alias("avg_price")
        ).withColumns({
            "lower_bound": col("q1") - 1.5 * (col("q3") - col("q1")),
            "upper_bound": col("q3") + 1.5 * (col("q3") - col("q1"))
        })
        
        # Join com dim_produtos para nome do produto
        produtos = spark.table("mvp_pucrio.gold_ecommerce.dim_produtos")
        
        # Join de volta para identificar outliers por produto
        joined = df.join(product_stats, "product_id", "left") \
                   .join(produtos, "product_id", "left")
        outliers = joined.filter(
            (col("unit_price") < col("lower_bound")) | (col("unit_price") > col("upper_bound"))
        )
        
        outlier_count = outliers.count()

        if outlier_count > 0:
            problems = 1
            print(f"\nOutliers detectados (por produto): {outlier_count} linhas")
            outliers.select(
                "product_id", "category", "order_id", "sales_value", "quantity", "unit_price", "avg_price",
                "median", "lower_bound", "upper_bound"
            ).orderBy(col("unit_price").desc()).limit(10).show(truncate=False)
        
    return problems

In [0]:
nulls_dict = dict()
duplicates_set = set()

print("\n" + "="*80)
print("QUALIDADE DE DADOS - GOLD LAYER")

problems = 0
schema = "mvp_pucrio.gold_ecommerce"
table_names = schema_tables(schema)

for name in table_names:
    full_name = f"{schema}.{name}"
    df = spark.table(full_name)

    print("="*80)
    print(f"\n\n🔍 Tabela: {name}")

    # print("\nVerificando completude e unicidade")
    num_nulls, problems, nulls_dict = find_nulls(df, pk_dict, columns_to_model, problems, nulls_dict)
    
    problems, duplicates_set = find_duplicates(df, pk_dict, columns_to_model, name, num_nulls, problems, duplicates_set)

    # print("\nVerificando consistência")
    if name in ("dim_sellers", "dim_customers"):
        problems += check_state(df, name)
    
    if name == "fato_vendas":
        problems += check_shipment(df, name)
    
    
    # print("\nVerificando acurácia")
    if name in ("dim_leads", "dim_converted_leads", "fato_vendas"):
        last_date = spark.table("mvp_pucrio.gold_ecommerce.dim_dates") \
    .agg(spark_max("order_date")).collect()[0][0]
        problems += check_temporal_accuracy(df, name, last_date)
    
    # print("\nVerificando outliers")
    if name == "fato_vendas":
        problems += detect_outliers(df, name)

if problems == 0:
    print(f"\n🎉 Nenhuma tabela com problemas detectados!")

### Análise

In [0]:
%sql
SELECT
    c.customer_state,
    ROUND(SUM(CASE WHEN d.year = 2016 THEN f.sales_value ELSE 0 END), 2) AS `2016_sales`,
    ROUND(SUM(CASE WHEN d.year = 2017 THEN f.sales_value ELSE 0 END), 2) AS `2017_sales`,
    ROUND(SUM(CASE WHEN d.year = 2018 THEN f.sales_value ELSE 0 END), 2) AS `2018_sales`,
    ROUND(SUM(f.sales_value), 2) AS total_sales_value
FROM mvp_pucrio.gold_ecommerce.dim_customers c
INNER JOIN mvp_pucrio.gold_ecommerce.fato_vendas f
    ON c.customer_id = f.customer_id
LEFT JOIN mvp_pucrio.gold_ecommerce.dim_dates d
    ON f.order_date = d.order_date
GROUP BY c.customer_state
ORDER BY total_sales_value DESC;


In [0]:
total_sales_monthly = spark.sql("""
    SELECT
        t.month, t.year,
        COUNT(DISTINCT f.order_id) AS num_orders,
        ROUND(SUM(f.sales_value), 2) AS total_sales_value
    FROM mvp_pucrio.gold_ecommerce.dim_dates t
    INNER JOIN mvp_pucrio.gold_ecommerce.fato_vendas f
        ON t.order_date = f.order_date
    GROUP BY t.month, t.year
    ORDER BY t.year, t.month
""")

df_monthly = total_sales_monthly.toPandas()

# Criar label month/year e ordenar
df_monthly['periodo'] = df_monthly['year'].astype(str) + '-' + df_monthly['month'].astype(str).str.zfill(2)
df_monthly = df_monthly.sort_values(['year', 'month']).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(df_monthly['periodo'], df_monthly['total_sales_value'])

ax.set_xlabel('Mes/Ano', fontsize=11)
ax.set_ylabel('Valor vendido (R$)', fontsize=11)
ax.set_title('Valor vendido por mês (R$)', fontsize=14, fontweight='bold')
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'R$ {x/1000:.0f}K'))
plt.xticks(rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

In [0]:
%sql
WITH sales_per_seller AS (
    SELECT
        seller_id,
        ROUND(SUM(f.sales_value), 2) AS total_sales,
        ROUND(SUM(f.commission), 2) AS total_commission,
        ROUND(SUM(f.sales_value) / COUNT(DISTINCT f.order_id), 2) AS avg_ticket
    FROM mvp_pucrio.gold_ecommerce.fato_vendas f
    WHERE f.order_date >= '2017-09-01'
    GROUP BY seller_id
)
SELECT 
    seller_id,
    total_sales,
    total_commission,
    avg_ticket,
    ROUND(total_sales * 100 / SUM(total_sales) OVER (), 2) AS percentage_of_total_sales,
    ROUND(
        SUM(total_sales) 
            OVER (ORDER BY total_sales DESC ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) * 100 / 
        SUM(total_sales) OVER (), 2
    ) AS cumm_percentage_of_total_sales
FROM sales_per_seller
ORDER BY total_sales DESC;

In [0]:
time_sales = spark.sql("""
    SELECT 
        time_category,
        COUNT(DISTINCT order_id) AS num_orders,
        ROUND(SUM(sales_value), 2) AS total_sold
    FROM mvp_pucrio.gold_ecommerce.fato_vendas
    WHERE order_date > '2017-09-01'
    GROUP BY time_category
    ORDER BY MIN(time_category)
""")

display(time_sales)


In [0]:
fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(time_sales.select('time_category').toPandas()['time_category'], time_sales.select('num_orders').toPandas()['num_orders'])

ax.set_xlabel('Intervalo de horas', fontsize=11)
ax.set_ylabel('Num. de ordens', fontsize=11)
ax.set_title('Número de ordens por intervalo de horas', fontsize=11)

plt.xticks(rotation=45, ha='right', fontsize=9)

plt.tight_layout()
plt.show()

In [0]:
%sql
SELECT 
    p.category,
    COUNT(DISTINCT f.order_id) AS num_orders,
    ROUND(SUM(f.sales_value), 2) AS total_sales,
    ROUND(AVG(f.sales_value), 2) AS avg_ticket,
    ROUND(SUM(f.commission), 2) AS total_commission
FROM mvp_pucrio.gold_ecommerce.fato_vendas f
LEFT JOIN mvp_pucrio.gold_ecommerce.dim_produtos p
    ON f.product_id = p.product_id
WHERE order_date > '2017-09-01'
GROUP BY p.category
ORDER BY total_sales DESC
LIMIT 20;

In [0]:
%sql
SELECT
    origin,
    COUNT(mql_id) AS leads_count,
    COUNT(seller_id) AS converted_leads_count,
    ROUND((COUNT(seller_id) / COUNT(mql_id)) * 100, 2) AS conversion_rate_percentage,
    ROUND(AVG(sales_cycle), 1) AS avg_sales_cycle_days,
    ROUND(MIN(sales_cycle), 1) AS min_cycle_days,
    ROUND(MAX(sales_cycle), 1) AS max_cycle_days
FROM mvp_pucrio.gold_ecommerce.dim_leads
GROUP BY origin
HAVING 
    origin != "unknown" AND
    origin != "other" AND
    origin != "other_publicities"
ORDER BY conversion_rate_percentage DESC;

In [0]:
%sql
WITH seller_metrics AS (
    SELECT 
        f.seller_id,
        COUNT(DISTINCT f.order_id) AS num_orders,
        ROUND(SUM(f.commission), 2) AS total_commission,
        MIN(f.order_date) AS first_sale_date,
        MAX(f.order_date) AS last_sale_date,
        DATEDIFF('2018-08-31', MAX(f.order_date)) AS days_since_last_sale,
        CASE 
            WHEN DATEDIFF('2018-08-31', MAX(f.order_date)) > 180 THEN 'Churned'
            ELSE 'Active'
        END AS seller_status
    FROM mvp_pucrio.gold_ecommerce.fato_vendas f
    WHERE f.order_date >= '2017-09-01'
    GROUP BY f.seller_id
)
SELECT 
    seller_status,
    COUNT(DISTINCT seller_id) AS num_sellers,
    ROUND((COUNT(DISTINCT seller_id) * 100.0 / SUM(COUNT(DISTINCT seller_id)) OVER ()), 2) AS percentage_sellers,
    ROUND(AVG(total_commission), 2) AS avg_ltv_commission,
    ROUND(AVG(num_orders), 2) AS avg_num_orders_per_seller,
    ROUND(AVG(days_since_last_sale), 1) AS avg_days_since_last_sale
FROM seller_metrics
GROUP BY seller_status
ORDER BY seller_status;